# Hydrological modeling using LSTMs

In [ ]:
# general
from tqdm import tqdm
from pathlib import Path
import xarray as xr
import pandas as pd
import shutil
import torch
import os

# hydrology
from neuralhydrology.nh_run import start_run

from ruamel.yaml import YAML
yaml = YAML()
yaml.preserve_quotes = True
yaml.indent(mapping=2, sequence=4, offset=2)
from src.utils import load_config, process_outputs, find_best_epoch
from src.lstm_inputs import prepare_lstm_inputs

cfg = load_config()
torch.cuda.is_available()

In [ ]:
experiment_name = "Whatif-Andes"
encoding = {'q_mm_sim': {"compression": "zstd"}}

## Model inputs

In [ ]:
# Build the attributes + per-basin time series in <pipeline>/lstm/data
prepare_lstm_inputs(cfg);

## Historical run

In [ ]:
# remove previous runs
[shutil.rmtree(Path(d)) for d in Path(cfg["dir"]["pipeline"] + "/lstm/runs").glob(f"{experiment_name}*")]

with open('modelling/hydrological_modelling_LSTM.yml') as stream:
    data = yaml.load(stream)

    # experiment name + data
    data['experiment_name'] = experiment_name
    data['data_dir']        = cfg["dir"]["pipeline"] + "/lstm/data"
    data['dynamic_inputs']  = ["prcp", "tas", "ep"]
    data['metrics']         = ["NSE", "KGE", "Pearson-r", "Beta-KGE", "Alpha-NSE"]
    data['epochs']          = 20

    # model selection
    data['model']            = "hybrid_model"
    data['conceptual_model'] = "SHM_glacier_snow_routing"

    data['duplicate_features']        = ['prcp', "ep", "tasmin", "tasmax", "gmelt_historical"]
    data["dynamic_conceptual_inputs"] = [f"{f}_copy1" for f in data['duplicate_features']]   
    data['custom_normalization']      = {k: {'centering': None, 'scaling': None} for k in 
                                         [f"{f}_copy1" for f in data['duplicate_features']] + ['q_mm']}   
    
    # hyperparameters from grid search
    data['batch_size']          = 2048
    data['hidden_size']         = 64
    data['learning_rate']       = {0: 1e-3, 10: 5e-4, 16: 1e-4}

    # sequence-to-sequence training: 730 d sequence, second half evaluated in the loss.
    # The first 365 d spin up both the LSTM hidden states and the SHM internal storages.
    # Do NOT set warmup_period: it slices x_conceptual too, which would remove the SHM
    # spin-up entirely and leave the storages initialised at their fixed defaults.
    data['seq_length']      = 270
    data['predict_last_n']  = 1

with open("modelling/hydrological_modelling_LSTM.yml", 'w') as stream:
    yaml.dump(data, stream)

# model run
torch.cuda.empty_cache()
start_run(config_file=Path("modelling/hydrological_modelling_LSTM.yml"))

In [ ]:
# set running dir IMPORTANT!!!!!!!!!!!!!!!!!!!!!!!!!
path_run = Path(cfg["dir"]["pipeline"]) / "lstm" / "runs"
path_run = next(path_run.glob(f"{experiment_name}*"))

# Switch the run config to predict_last_n=1 for all inference below.
# predict_last_n only controls loss subsetting (training) and output slicing (evaluation) -
# it is not part of the architecture, so the trained weights are untouched. For a single
# frequency the tester keeps only the last step of each sequence regardless, so metrics are
# bit-identical, but memory drops from ~25 GB to ~0.1 GB over the 962 target basins.
with open(path_run / "config.yml") as stream:
    run_cfg = yaml.load(stream)
    run_cfg['predict_last_n'] = 1

with open(path_run / "config.yml", 'w') as stream:
    yaml.dump(run_cfg, stream)

In [ ]:
# Find best epoch
best_epoch = find_best_epoch(path_run, n_epoch=data['epochs'], metric="KGE")

# Save time series data + performance metrics (andeangc)
ts_list = []
metrics_list = []

for split in ["train", "validation", "test"]:
    ts_period, metrics_split = process_outputs(path_run, split, best_epoch, metrics=data['metrics'])
    metrics_split["split"] = split
    ts_list.append(ts_period)
    metrics_list.append(metrics_split)

ts_list = xr.merge(ts_list)
ts_list.to_netcdf("data/hydro/runoff_sim_andeangc.nc", encoding=encoding)

metrics_list = pd.concat(metrics_list)
metrics_list.to_csv("data/hydro/performance_metrics.csv")

In [ ]:
# save time series (hydroatlas)
with open(path_run / "config.yml") as stream:
    data = yaml.load(stream)
    data['test_basin_file'] = "modelling/basins_test_target.txt"

with open(path_run / "config.yml", 'w') as stream:
    yaml.dump(data, stream)

ts_period = process_outputs(path_run, "test", best_epoch, metrics=None)
ts_period.to_netcdf("data/hydro/runoff_sim_hydroatlas_hist.nc", encoding=encoding)

## Mass conservation test

In [ ]:
basins = list(Path(cfg["dir"]["pipeline"]).glob("lstm/data/time_series/*nc"))
basins = [b for b in basins if b.name.endswith('.nc') and b.stem.isdigit()]

for basin_path in tqdm(basins): 
        
        # Load data into memory
        basin = xr.open_dataset(basin_path)
        basin = basin.load()  # Load into memory
        basin.close()  # Close file handle
        
        # Modify data
        basin["gmelt_historical"] = 0
        
        # Save to file
        basin.to_netcdf(basin_path)

ts_whatif = process_outputs(path_run, "test", best_epoch, metrics=None)
ts_whatif.to_netcdf("data/hydro/runoff_sim_hydroatlas_mc.nc", encoding=encoding) 

## What-if runs

In [ ]:
# what-if scenarios
cmip6_models = pd.read_csv("data/climate/cmip6_models_selection.csv")

df_whatif = []
for idx, row in cmip6_models.iterrows():

    group = row["group"]
    gcm   = row["model"]
    ssp   = row["exp"]

    # TODO maybe this can be improved by using the config file directly
    basins = list(Path(cfg["dir"]["pipeline"]).glob("lstm/data/time_series/*nc"))
    basins = [b for b in basins if b.name.endswith('.nc') and b.stem.isdigit()]

    for basin_path in tqdm(basins): 
        # Load data into memory
        basin = xr.open_dataset(basin_path)
        basin = basin.load()  # Load into memory
        basin.close()  # Close file handle
        
        # Modify data
        basin["gmelt_historical"] = basin["gmelt_historical_{}_{}_{}".format(group, gcm, ssp)]
        
        # Save to file
        basin.to_netcdf(basin_path)
        
    ts_whatif = process_outputs(path_run, "test", best_epoch, metrics=None)
    df_whatif.append(ts_whatif)

df_whatif = xr.concat(df_whatif, dim='whatif_scenario')
df_whatif = df_whatif.assign_coords(whatif_scenario=("whatif_scenario", cmip6_models.group)) 
df_whatif.to_netcdf("data/hydro/runoff_sim_hydroatlas_whatif.nc", encoding=encoding) 